In [2]:
!pip install openpyxl


   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpy


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
import os
import pandas as pd
import numpy as np
from datetime import datetime

# ---------------------------------------------------------
# Step 1: File Paths & Folders Setup
# ---------------------------------------------------------
RAW_DATA_PATH = "data/raw_data.csv"
PROCESSED_DATA_PATH = "data/cleaned_dataset.csv"
EXCEL_OUTPUT_PATH = "reports/KPI_Summary_Report.xlsx"

os.makedirs("data", exist_ok=True)
os.makedirs("reports", exist_ok=True)

def run_pipeline():
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] Starting Automation Pipeline...")

    # ---------------------------------------------------------
    # Step 2: Load Raw Data
    # ---------------------------------------------------------
    if not os.path.exists(RAW_DATA_PATH):
        print("Raw data file not found. Creating sample dataset...")
        df_raw = pd.DataFrame({
            "Customer_ID": [101, 102, 103, 104, 105, 105, 106, 107, 108],
            "Order_Date": ["2026-01-10", "2026-01-12", None, "2026-02-01", "2026-02-15", "2026-02-15", "2026-03-01", "2026-03-10", "2026-03-20"],
            "Revenue": [250.0, 450.5, 120.0, np.nan, 800.0, 800.0, 310.0, 150.0, 950.0],
            "Status": ["Completed", "Completed", "Pending", "Cancelled", "Completed", "Completed", "Completed", "Refunded", "Completed"],
            "Days_Inactive": [12, 45, 60, 5, 20, 20, 90, 15, 8]
        })
        df_raw.to_csv(RAW_DATA_PATH, index=False)
    else:
        df_raw = pd.read_csv(RAW_DATA_PATH)
    
    print(f"Raw data loaded: {len(df_raw)} records.")

    # ---------------------------------------------------------
    # Step 3: Data Cleaning
    # ---------------------------------------------------------
    df = df_raw.copy()

    # 1. Remove duplicate records
    initial_count = len(df)
    df = df.drop_duplicates()
    print(f"Removed {initial_count - len(df)} duplicate record(s).")

    # 2. Fill missing values
    df["Revenue"] = df["Revenue"].fillna(df["Revenue"].median())
    df["Order_Date"] = df["Order_Date"].fillna("2026-01-01")

    # 3. Set correct data types
    df["Order_Date"] = pd.to_datetime(df["Order_Date"])
    df["Customer_ID"] = df["Customer_ID"].astype(int)

    # ---------------------------------------------------------
    # Step 4: Save Cleaned Data
    # ---------------------------------------------------------
    df.to_csv(PROCESSED_DATA_PATH, index=False)
    print(f"Cleaned dataset saved at: {PROCESSED_DATA_PATH}")

    # ---------------------------------------------------------
    # Step 5: Calculate Key KPIs
    # ---------------------------------------------------------
    completed_orders = df[df["Status"] == "Completed"]
    total_revenue = completed_orders["Revenue"].sum()
    total_orders_count = len(completed_orders)
    active_customers = df["Customer_ID"].nunique()
    avg_order_value = total_revenue / total_orders_count if total_orders_count > 0 else 0
    at_risk_customers = len(df[df["Days_Inactive"] > 45])
    churn_rate = (at_risk_customers / active_customers) * 100 if active_customers > 0 else 0

    kpi_summary = pd.DataFrame({
        "KPI Metric": [
            "Total Revenue ($)",
            "Completed Orders",
            "Active Customers",
            "Average Order Value ($)",
            "At-Risk Customers (>45 Days Inactive)",
            "Estimated Churn Rate (%)"
        ],
        "Value": [
            round(total_revenue, 2),
            total_orders_count,
            active_customers,
            round(avg_order_value, 2),
            at_risk_customers,
            f"{round(churn_rate, 2)}%"
        ]
    })

    status_breakdown = df.groupby("Status").agg(
        Order_Count=("Customer_ID", "count"),
        Total_Revenue=("Revenue", "sum")
    ).reset_index()

    # ---------------------------------------------------------
    # Step 6: Export to Excel File
    # ---------------------------------------------------------
    with pd.ExcelWriter(EXCEL_OUTPUT_PATH, engine="openpyxl") as writer:
        kpi_summary.to_excel(writer, sheet_name="KPI Summary", index=False)
        status_breakdown.to_excel(writer, sheet_name="Status Breakdown", index=False)
        df.to_excel(writer, sheet_name="Cleaned Data", index=False)

    print(f"Excel report successfully created at: {EXCEL_OUTPUT_PATH}")
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] Pipeline executed successfully!\n")

if __name__ == "__main__":
    run_pipeline()

[2026-07-29 11:44:55] Starting Automation Pipeline...
Raw data loaded: 9 records.
Removed 1 duplicate record(s).
Cleaned dataset saved at: data/cleaned_dataset.csv
Excel report successfully created at: reports/KPI_Summary_Report.xlsx
[2026-07-29 11:44:55] Pipeline executed successfully!

